# **vae-1d-pixelwise — training**

Self-contained training notebook for **vae-1d-pixelwise** across the 3 datasets (IIRS / M3 / AVIRIS).\n\nTrained under **both** loss regimes per dataset (standard = MSE+βKLD, physics = +λ·SAM), saved as `model/<DATASET>/<model>_<loss>.pt`.

## **Config**

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path as _Path
import yaml as _yaml

# --------------------------------------------------------------------------
# Datasets in the ablation. Edit DATA_ROOTS to point at your processed patches
# (each root should contain <scene>/<split>/patch_*.npy). On Kaggle/Colab these
# will be /kaggle/input/... paths; locally they default to data/processed/<DS>.
# --------------------------------------------------------------------------
DATASETS = {
    "IIRS":   {"input_channels": 256},
    "M3":     {"input_channels": 84},
    "AVIRIS": {"input_channels": 424},
}

DATA_ROOTS = {
    "IIRS":   "data/processed/IIRS",
    "M3":     "data/processed/M3",
    "AVIRIS": "data/processed/AVIRIS",
}

# Where trained checkpoints are written: CKPT_ROOT/<DATASET>/<name>.pt
CKPT_ROOT = "model"

# Where the per-dataset hyperparam YAMLs live (repo-root-relative).
HYPERPARAM_CONFIG_DIR = "utils/hyperparam_configs"


@dataclass
class Settings:
    input_height: int = 64
    input_width: int = 64
    input_channels: int = 256          # overridden per dataset via make_settings()

    # training
    batch_size: int = 4
    num_workers: int = 4
    epochs: int = 20
    lr: float = 5e-3
    beta: float = 1e-3
    lambda_physics: float = 0.3

    # spatial branch
    reduced_dims: int = 32
    latent_dim: int = 256
    n_2D_conv_blocks: int = 4
    conv2D_kernel_size: int = 3
    conv_output_c: int = field(init=False)
    conv_output_h: int = field(init=False)
    conv_output_w: int = field(init=False)

    # spectral branch
    spectral_n_1D_conv_blocks: int = 2
    spectral_conv1D_kernel_size: int = 4
    spectral_latent_dim: int = 128
    spectral_linear_expansion_dim: int = field(init=False)
    spectral_transpose_c: int = field(init=False)
    spectral_transpose_l: int = field(init=False)

    # Baseline capacity knobs (overridden per-dataset by hyperparam YAML so each
    # baseline matches vae-our's param count at that dataset). IIRS defaults.
    vae_standard_base_ch: int = 134
    vae_standard_n_down: int = 3
    vae_standard_latent_ch: int = 16

    vae_3d_base_ch: int = 78
    vae_3d_n_down: int = 3
    vae_3d_latent_ch: int = 8

    vae_1d_hidden_dims: tuple = (4224, 2112, 1056)
    vae_1d_latent_dim: int = 32

    def __post_init__(self):
        self.conv_output_c = self.reduced_dims * (2 ** self.n_2D_conv_blocks)
        self.conv_output_h = self.input_height // (2 ** self.n_2D_conv_blocks)
        self.conv_output_w = self.input_width // (2 ** self.n_2D_conv_blocks)
        self.spectral_transpose_c = self.input_channels * (2 ** (self.spectral_n_1D_conv_blocks - 1))
        self.spectral_transpose_l = self.input_channels // (2 ** self.spectral_n_1D_conv_blocks)
        self.spectral_linear_expansion_dim = self.spectral_transpose_c * self.spectral_transpose_l


def make_settings(dataset):
    """Return a Settings whose band count matches the dataset."""
    return Settings(input_channels=DATASETS[dataset]["input_channels"])


# --------------------------------------------------------------------------
# Per-dataset hyperparam loader — mirrors utils/hyperparams.py but re-implemented
# locally so the notebook stays self-contained.
# --------------------------------------------------------------------------
_HP_SETTINGS_FIELDS = {
    "batch_size", "num_workers",
    "vae_standard_base_ch", "vae_standard_n_down", "vae_standard_latent_ch",
    "vae_3d_base_ch", "vae_3d_n_down", "vae_3d_latent_ch",
    "vae_1d_hidden_dims", "vae_1d_latent_dim",
    # notebook Settings also carries these as fields (unlike utils/config.py Settings):
    "epochs", "lr", "beta", "lambda_physics",
}
_HP_OPTIMIZATION_FIELDS = {"seed", "weight_decay", "early_stopping_patience"}
_HP_ALLOWED = _HP_SETTINGS_FIELDS | _HP_OPTIMIZATION_FIELDS


def load_hyperparams(dataset, config_dir=HYPERPARAM_CONFIG_DIR):
    """Load hyperparam-config-<DATASET>.yaml -> dict. Empty dict if missing."""
    path = _Path(config_dir) / f"hyperparam-config-{dataset.upper()}.yaml"
    if not path.exists():
        return {}
    with open(path) as f:
        raw = _yaml.safe_load(f) or {}
    unknown = set(raw) - _HP_ALLOWED
    if unknown:
        raise ValueError(f"{path}: unknown hyperparam keys: {sorted(unknown)}")
    return raw


def apply_hyperparams(settings, hp):
    """Mutate `settings` in place for whitelisted fields present in `hp`."""
    for key in _HP_SETTINGS_FIELDS & set(hp):
        value = hp[key]
        if key == "vae_1d_hidden_dims" and isinstance(value, list):
            value = tuple(value)
        setattr(settings, key, value)


# Global settings object the branch classes read from. Reassigned per dataset
# inside the training loop (rebuild the model after reassigning).
settings = make_settings("IIRS")


## **Imports**

In [ ]:
import math
import time
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch import Tensor
from torch.nn import Conv2d, ConvTranspose2d
from torch.utils.data import Dataset, DataLoader

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


## **Loss (SAM physics prior + KL)**

In [ ]:
def spectral_angle_mapper_loss(y_true, y_pred):
    """Differentiable physics prior: mean spectral angle (radians)."""
    dot = torch.sum(y_true * y_pred, dim=-1)
    nt = torch.sqrt(torch.sum(y_true ** 2, dim=-1) + 1e-8)
    npd = torch.sqrt(torch.sum(y_pred ** 2, dim=-1) + 1e-8)
    cos = torch.clamp(dot / (nt * npd + 1e-8), -1.0 + 1e-8, 1.0 - 1e-8)
    return torch.mean(torch.acos(cos))

In [ ]:
def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

## **Metrics (PSNR / SSIM)**

In [ ]:
def compute_psnr(img1, img2, data_range=1.0):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return float("inf")
    return (20 * torch.log10(torch.tensor(data_range).to(img1.device)) - 10 * torch.log10(mse)).item()

In [ ]:
def compute_ssim(img1, img2, data_range=1.0, window_size=11):
    if img1.dim() == 4 and img1.shape[-1] not in [img1.shape[1], img1.shape[2]]:
        img1 = img1.permute(0, 3, 1, 2); img2 = img2.permute(0, 3, 1, 2)
    channels = img1.shape[1]

    def gaussian(w, sigma):
        g = torch.exp(torch.tensor([-(x - w // 2) ** 2 / (2 * sigma ** 2) for x in range(w)]))
        return g / g.sum()

    _1d = gaussian(window_size, 1.5).unsqueeze(1).to(img1.device)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2d.expand(channels, 1, window_size, window_size).contiguous()
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channels)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channels)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    s1 = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channels) - mu1_sq
    s2 = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channels) - mu2_sq
    s12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channels) - mu1_mu2
    c1 = (0.01 * data_range) ** 2; c2 = (0.03 * data_range) ** 2
    ssim = ((2 * mu1_mu2 + c1) * (2 * s12 + c2)) / ((mu1_sq + mu2_sq + c1) * (s1 + s2 + c2))
    return ssim.mean().item()


## **Data loader**

In [ ]:
class HSIPatchDataset(Dataset):
    """All .npy patches for a split, max-normalized to [0, 1] on the fly."""
    def __init__(self, processed_root, split):
        assert split in ("train", "valid", "test")
        self.patch_files: List[Path] = sorted(Path(processed_root).glob(f"**/{split}/*.npy"))

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        patch = np.load(self.patch_files[idx], mmap_mode="r").astype(np.float32)
        m = patch.max()
        if m > 0:
            patch = patch / m
        return torch.from_numpy(patch)


def build_dataloader(processed_root, split, batch_size=None, shuffle=True, num_workers=None, pin_memory=True):
    ds = HSIPatchDataset(processed_root, split)
    return DataLoader(
        ds,
        batch_size=batch_size or settings.batch_size,
        shuffle=shuffle,
        num_workers=num_workers if num_workers is not None else settings.num_workers,
        pin_memory=pin_memory,
        drop_last=(split == "train"),
    )


## **Model definition**

MODEL DEFINITION  --  vae-1d-pixelwise  (Baseline C: 1D Pixel-Wise VAE)
Per-pixel MLP VAE (Su et al., 2019, deep-autoencoder unmixing): the (B,H,W)
grid is folded entirely into the batch dim, so each pixel spectrum is encoded
independently. Excellent chemistry (low SAM) but no spatial context to denoise
corrupted pixels (poor PSNR/SSIM).

In [ ]:
class VAE_1D_Pixelwise(nn.Module):
    def __init__(self):
        super().__init__()
        c = settings.input_channels
        hidden_dims = tuple(settings.vae_1d_hidden_dims)
        latent_dim = settings.vae_1d_latent_dim
        self.latent_dim = latent_dim
        enc, in_f = [], c
        for h in hidden_dims:
            enc += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        enc.append(nn.Linear(in_f, 2 * latent_dim))
        self.encoder = nn.Sequential(*enc)
        dec, in_f = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        dec.append(nn.Linear(in_f, c))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=-1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        b, h, w, c = x.shape
        z, mu, logvar = self.reparameterize(self.encoder(x.reshape(b * h * w, c)))
        recon = torch.sigmoid(self.decoder(z)).reshape(b, h, w, c)
        mu = mu.reshape(b, h, w, self.latent_dim)
        logvar = logvar.reshape(b, h, w, self.latent_dim)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        b, h, w, c = x.shape
        mu, _ = torch.chunk(self.encoder(x.reshape(b * h * w, c)), 2, dim=-1)
        return [mu.reshape(b, h, w, self.latent_dim)]

    @torch.no_grad()
    def decode_latents(self, latents):
        z = latents[0]; b, h, w, _ = z.shape
        recon = torch.sigmoid(self.decoder(z.reshape(b * h * w, self.latent_dim)))
        return recon.reshape(b, h, w, settings.input_channels)


def compute_losses(model, x, beta, lambda_physics, use_physics=False):
    recon, mu, logvar = model(x)
    mse = F.mse_loss(recon, x)
    kld = kl_divergence(mu, logvar)
    sam = spectral_angle_mapper_loss(x, recon)
    loss = mse + beta * kld + (lambda_physics * sam if use_physics else 0.0)
    return loss, mse, sam, kld, recon


def build_model():
    return VAE_1D_Pixelwise()


## **Training loop**

In [ ]:
def train_vae(model, dataloader, epochs, device, use_physics, lr, beta, lambda_physics,
              val_dataloader=None, ckpt_file=None, ckpt_meta=None,
              weight_decay=1e-5, patience=7):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.to(device)
    ckpt_meta = ckpt_meta or {}
    best = math.inf
    no_improve = 0
    if ckpt_file:
        Path(ckpt_file).parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train(); tl = tm = ts = tk = tp = tss = 0.0
        for x in dataloader:
            x = x.to(device); optimizer.zero_grad()
            loss, mse, sam, kld, recon = compute_losses(model, x, beta, lambda_physics, use_physics)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            tl += loss.item(); tm += mse.item(); ts += sam.item(); tk += kld.item()
            tp += compute_psnr(x, recon); tss += compute_ssim(x, recon)
        n = max(len(dataloader), 1)
        tl, tm, ts, tk, tp, tss = tl/n, tm/n, ts/n, tk/n, tp/n, tss/n

        vl = vm = vs = vk = vp = vss = 0.0
        if val_dataloader is not None:
            model.eval()
            with torch.no_grad():
                for x in val_dataloader:
                    x = x.to(device)
                    loss, mse, sam, kld, recon = compute_losses(model, x, beta, lambda_physics, use_physics)
                    vl += loss.item(); vm += mse.item(); vs += sam.item(); vk += kld.item()
                    vp += compute_psnr(x, recon); vss += compute_ssim(x, recon)
            nv = max(len(val_dataloader), 1)
            vl, vm, vs, vk, vp, vss = vl/nv, vm/nv, vs/nv, vk/nv, vp/nv, vss/nv
        scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] Loss {tl:.4f} MSE {tm:.4f} SAM {ts:.4f} "
              f"KLD {tk:.4f} PSNR {tp:.2f} SSIM {tss:.4f}"
              + (f" | Val Loss {vl:.4f} PSNR {vp:.2f} SSIM {vss:.4f}" if val_dataloader else ""))

        monitor = vl if val_dataloader is not None else tl
        if monitor < best:
            best = monitor
            no_improve = 0
            if ckpt_file:
                torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(), "loss": monitor,
                            **ckpt_meta}, ckpt_file)
        else:
            no_improve += 1

        if val_dataloader is not None and no_improve >= patience:
            print(f"Early stopping at epoch {epoch}: no val_loss improvement for {patience} epochs.")
            break

    if ckpt_file:
        print("Saved best checkpoint to", ckpt_file)


## **Train across all datasets × loss regimes**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "vae-1d-pixelwise"   # trained twice per dataset: standard + physics

for ds in ["IIRS", "M3", "AVIRIS"]:
    for loss_type in ["standard", "physics"]:
        print("\n" + "=" * 60)
        print(f" TRAIN {MODEL_NAME} | {ds} | {loss_type}")
        print("=" * 60)

        settings = make_settings(ds)
        hp = load_hyperparams(ds)
        apply_hyperparams(settings, hp)
        globals()["settings"] = settings
        patience = hp.get("early_stopping_patience", 7)
        weight_decay = hp.get("weight_decay", 1e-5)

        train_loader = build_dataloader(DATA_ROOTS[ds], "train", shuffle=True)
        val_loader   = build_dataloader(DATA_ROOTS[ds], "valid", shuffle=False)
        print(f"  C={settings.input_channels} | train batches {len(train_loader)} | val batches {len(val_loader)}")

        model = build_model().to(device)   # <-- defined in the MODEL DEFINITION cell above
        with torch.no_grad():
            model(torch.randn(2, settings.input_height, settings.input_width, settings.input_channels, device=device))

        ckpt_file = Path(CKPT_ROOT) / ds / f"{MODEL_NAME}_{loss_type}.pt"
        train_vae(model, train_loader, settings.epochs, device, use_physics=(loss_type == "physics"),
                  lr=settings.lr, beta=settings.beta, lambda_physics=settings.lambda_physics,
                  val_dataloader=val_loader, ckpt_file=ckpt_file,
                  ckpt_meta={"model": MODEL_NAME, "dataset": ds, "loss_type": loss_type},
                  weight_decay=weight_decay, patience=patience)
